# QQQ Daily ICT Analysis

Fetch daily QQQ candles from Schwab, run ICT analysis, and visualize setups.

**ICT Concepts covered:**
- Market Structure (Swing Highs/Lows, BOS, CHoCH)
- Order Blocks (Bullish/Bearish)
- Fair Value Gaps (FVGs)
- Liquidity Levels (Equal Highs/Lows)
- Displacement Candles

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import pandas as pd
import numpy as np
import mplfinance as mpf
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from datetime import datetime
import pytz

from qqq_ingest.schwab import SchwabDailyFetcher
from qqq_ingest.ict import run_ict_analysis, signals

TZ_ET = pytz.timezone('US/Eastern')
print('Imports OK')

## 1. Fetch Daily Candles

In [ ]:
# Fetch from Schwab (requires valid tokens.json)
from dateutil.relativedelta import relativedelta

end = datetime.now(TZ_ET)
start = end - relativedelta(months=18)

fetcher = SchwabDailyFetcher()
df_raw = fetcher.fetch('QQQ', start, end)
print(f'{len(df_raw)} daily candles: {df_raw["date"].min()} → {df_raw["date"].max()}')
df_raw.tail()

In [ ]:
# Or load from saved parquet if you already ran fetch_daily.py
# df_raw = pd.read_parquet('../data/daily_candles.parquet')
# print(f'{len(df_raw)} daily candles loaded')

## 2. Run ICT Analysis

In [ ]:
df = run_ict_analysis(df_raw, lookback=3)
print(f'Analysis complete. Columns: {list(df.columns)}')

# Recent signals
sig = signals(df, last_n=20)
if not sig.empty:
    print(f'\n--- Recent ICT Signals (last 20 bars) ---')
    for _, row in sig.iterrows():
        print(f"  {row['date']}  close={row['close']:.2f}  {row['signals']}")
else:
    print('No signals in last 20 bars.')

## 3. Annotated Candlestick Chart

Shows the last N days with ICT markings overlaid.

In [ ]:
def plot_ict(df: pd.DataFrame, last_n: int = 60, title: str = 'QQQ Daily – ICT Analysis'):
    """Plot annotated candlestick chart with ICT structures."""
    chunk = df.tail(last_n).copy()
    chunk['date'] = pd.to_datetime(chunk['date'])
    chunk = chunk.set_index('date')
    chunk.index.name = 'Date'

    # Prepare mplfinance OHLCV
    ohlcv = chunk[['open', 'high', 'low', 'close', 'volume']].copy()
    ohlcv.columns = ['Open', 'High', 'Low', 'Close', 'Volume']

    # Build addplots for swing points
    ap = []

    # Swing highs as triangles above
    sh_marker = np.where(chunk['swing_high'], chunk['high'] * 1.002, np.nan)
    ap.append(mpf.make_addplot(sh_marker, type='scatter', marker='v', markersize=60, color='red'))

    # Swing lows as triangles below
    sl_marker = np.where(chunk['swing_low'], chunk['low'] * 0.998, np.nan)
    ap.append(mpf.make_addplot(sl_marker, type='scatter', marker='^', markersize=60, color='green'))

    # Displacement candles
    bull_disp = np.where(chunk['displacement'] == 'bullish_disp', chunk['low'] * 0.995, np.nan)
    bear_disp = np.where(chunk['displacement'] == 'bearish_disp', chunk['high'] * 1.005, np.nan)
    ap.append(mpf.make_addplot(bull_disp, type='scatter', marker='D', markersize=40, color='lime'))
    ap.append(mpf.make_addplot(bear_disp, type='scatter', marker='D', markersize=40, color='magenta'))

    style = mpf.make_mpf_style(base_mpf_style='charles', rc={'font.size': 9})

    fig, axes = mpf.plot(
        ohlcv, type='candle', volume=True, style=style,
        addplot=ap, title=title,
        figsize=(18, 9), returnfig=True,
    )
    ax = axes[0]

    # Overlay FVG zones as shaded rectangles
    dates = list(ohlcv.index)
    for i, (idx, row) in enumerate(chunk.iterrows()):
        if row.get('fvg_type') == 'bullish_fvg':
            ax.axhspan(row['fvg_bottom'], row['fvg_top'], xmin=i/len(dates),
                       xmax=min((i+5)/len(dates), 1.0), alpha=0.15, color='green')
        elif row.get('fvg_type') == 'bearish_fvg':
            ax.axhspan(row['fvg_bottom'], row['fvg_top'], xmin=i/len(dates),
                       xmax=min((i+5)/len(dates), 1.0), alpha=0.15, color='red')

    # Overlay Order Block zones
    for i, (idx, row) in enumerate(chunk.iterrows()):
        if row.get('ob_type') == 'bullish_ob':
            ax.axhspan(row['ob_bottom'], row['ob_top'], xmin=i/len(dates),
                       xmax=min((i+8)/len(dates), 1.0), alpha=0.10, color='blue')
        elif row.get('ob_type') == 'bearish_ob':
            ax.axhspan(row['ob_bottom'], row['ob_top'], xmin=i/len(dates),
                       xmax=min((i+8)/len(dates), 1.0), alpha=0.10, color='orange')

    # Annotate BOS/CHoCH
    for i, (idx, row) in enumerate(chunk.iterrows()):
        evt = row.get('structure_event')
        if evt:
            label = evt.replace('_', ' ').upper()
            color = 'green' if 'bullish' in evt else 'red'
            y = row['high'] * 1.008 if 'bullish' in evt else row['low'] * 0.992
            ax.annotate(label, xy=(i, y), fontsize=7, color=color,
                        ha='center', fontweight='bold', rotation=45)

    # Mark equal highs/lows with horizontal dashed lines
    for i, (idx, row) in enumerate(chunk.iterrows()):
        if row.get('eq_high'):
            ax.axhline(y=row['high'], color='red', linestyle='--', alpha=0.4, linewidth=0.8)
        if row.get('eq_low'):
            ax.axhline(y=row['low'], color='green', linestyle='--', alpha=0.4, linewidth=0.8)

    plt.tight_layout()
    plt.show()


plot_ict(df, last_n=60)

## 4. Zoom In – Last 20 Days

In [ ]:
plot_ict(df, last_n=20, title='QQQ Daily – ICT (Last 20 Days)')

## 5. Active Zones

Show unmitigated order blocks and open FVGs that price hasn't returned to yet.

In [ ]:
def active_zones(df: pd.DataFrame, zone_type: str = 'ob') -> pd.DataFrame:
    """
    Find unmitigated OBs or unfilled FVGs.
    zone_type: 'ob' for order blocks, 'fvg' for fair value gaps.
    """
    if zone_type == 'ob':
        type_col, top_col, bot_col = 'ob_type', 'ob_top', 'ob_bottom'
    else:
        type_col, top_col, bot_col = 'fvg_type', 'fvg_top', 'fvg_bottom'

    zones = df[df[type_col].notna()][['date', type_col, top_col, bot_col]].copy()
    current_price = df['close'].iloc[-1]
    active = []

    for _, z in zones.iterrows():
        zone_date_idx = df[df['date'] == z['date']].index[0]
        subsequent = df.iloc[zone_date_idx + 1:]

        # Check if price has traded through the zone (mitigated)
        if 'bullish' in z[type_col]:
            mitigated = (subsequent['low'] <= z[bot_col]).any()
        else:
            mitigated = (subsequent['high'] >= z[top_col]).any()

        if not mitigated:
            dist = ((z[top_col] + z[bot_col]) / 2 - current_price) / current_price * 100
            active.append({
                'date': z['date'],
                'type': z[type_col],
                'top': z[top_col],
                'bottom': z[bot_col],
                'distance_%': round(dist, 2),
            })

    return pd.DataFrame(active)


print('=== Unmitigated Order Blocks ===')
active_obs = active_zones(df, 'ob')
if not active_obs.empty:
    print(active_obs.to_string(index=False))
else:
    print('None found.')

print('\n=== Unfilled Fair Value Gaps ===')
active_fvgs = active_zones(df, 'fvg')
if not active_fvgs.empty:
    print(active_fvgs.to_string(index=False))
else:
    print('None found.')

## 6. Bias Summary

Quick directional read based on current market structure.

In [ ]:
def bias_summary(df: pd.DataFrame):
    """Print a quick directional bias based on recent structure."""
    last = df.iloc[-1]
    price = last['close']

    # Find most recent structure event
    struct_events = df[df['structure_event'].notna()].tail(3)
    if struct_events.empty:
        print('No recent structure events — no clear bias.')
        return

    latest_event = struct_events.iloc[-1]
    event = latest_event['structure_event']

    print(f'Current price: {price:.2f}')
    print(f'Last structure event: {event} on {latest_event["date"]}')
    print()

    if 'bullish' in event:
        print('BIAS: BULLISH')
        print('Look for: pullbacks into bullish OBs / FVGs for long entries')
        # Find nearest bullish OB below price
        bull_obs = active_zones(df, 'ob')
        bull_obs = bull_obs[bull_obs['type'] == 'bullish_ob']
        bull_obs = bull_obs[bull_obs['top'] < price]
        if not bull_obs.empty:
            nearest = bull_obs.iloc[-1]
            print(f'Nearest bullish OB below: {nearest["bottom"]:.2f}–{nearest["top"]:.2f} ({nearest["date"]})')

        bull_fvgs = active_zones(df, 'fvg')
        bull_fvgs = bull_fvgs[bull_fvgs['type'] == 'bullish_fvg']
        bull_fvgs = bull_fvgs[bull_fvgs['top'] < price]
        if not bull_fvgs.empty:
            nearest = bull_fvgs.iloc[-1]
            print(f'Nearest bullish FVG below: {nearest["bottom"]:.2f}–{nearest["top"]:.2f} ({nearest["date"]})')
    else:
        print('BIAS: BEARISH')
        print('Look for: rallies into bearish OBs / FVGs for short entries')
        bear_obs = active_zones(df, 'ob')
        bear_obs = bear_obs[bear_obs['type'] == 'bearish_ob']
        bear_obs = bear_obs[bear_obs['top'] > price]
        if not bear_obs.empty:
            nearest = bear_obs.iloc[0]
            print(f'Nearest bearish OB above: {nearest["bottom"]:.2f}–{nearest["top"]:.2f} ({nearest["date"]})')

        bear_fvgs = active_zones(df, 'fvg')
        bear_fvgs = bear_fvgs[bear_fvgs['type'] == 'bearish_fvg']
        bear_fvgs = bear_fvgs[bear_fvgs['top'] > price]
        if not bear_fvgs.empty:
            nearest = bear_fvgs.iloc[0]
            print(f'Nearest bearish FVG above: {nearest["bottom"]:.2f}–{nearest["top"]:.2f} ({nearest["date"]})')

    # Recent structure context
    print(f'\nRecent structure events:')
    for _, row in struct_events.iterrows():
        print(f'  {row["date"]}  {row["structure_event"]}')


bias_summary(df)